# Notebook 05 — Resiliência acadêmica

São considerados **socioeconomicamente desfavorecidos** os estudantes no quartil inferior do Indicador de Nível Socioeconômico (INSE) calculado pelo INEP, e **academicamente resilientes** os desfavorecidos cuja proficiência em Língua Portuguesa está no quartil superior.

Referência dos quartis. Os quartis são calculados na própria base analítica (estudantes da série final do Ensino Médio de escolas públicas de Mato Grosso, SAEB 2023). A resiliência aqui é relativa a esta população.

| Seção | Conteúdo |
| --- | --- |
| 1 | Quartis e regra para empates |
| 2 | Resiliência observada e esperada |
| 3 | Sensibilidade da definição |
| 4 | Taxas de resiliência por raça/cor e por sexo |
| 5 | Perfil dos estudantes resilientes |
| 6 | Modelo complementar |
**Entrada:** `base_analitica.parquet`. **Saída:** `tabelas_resiliencia.xlsx`.

## 0. Configuração

In [1]:
import sys, numpy as np, pandas as pd
import statsmodels, statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests
import warnings; warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/dissertacao/'   # ajuste para a sua pasta
except ImportError:
    PASTA = './'

SEMENTE = 2025
FATOR_SAEB = 55.093
print({'python': sys.version.split()[0], 'pandas': pd.__version__, 'statsmodels': statsmodels.__version__})
base = pd.read_parquet(PASTA + 'base_analitica.parquet').reset_index(drop=True)
assert len(base) == 22996 and base['inse'].notna().all()
TABELAS = {}

def fmt_p(p):
    return '< 0,001' if p < 0.001 else f'{p:.3f}'.replace('.', ',')

def ic_escolas(ind, grupos, B=1000, seed=SEMENTE):
    """Intervalo de 95% para uma proporção por reamostragem de escolas (considera o agrupamento)."""
    df = pd.DataFrame({'x': ind, 'g': grupos}).groupby('g')['x'].agg(['sum', 'count'])
    rng = np.random.default_rng(seed)
    est = [df.iloc[a]['sum'].sum() / df.iloc[a]['count'].sum()
           for a in (rng.integers(0, len(df), len(df)) for _ in range(B))]
    return np.percentile(est, [2.5, 97.5])

Mounted at /content/drive
{'python': '3.13.16', 'pandas': '2.2.3', 'statsmodels': '0.15.0'}


## 1. Quartis e regra para empates

O INSE tem poucos valores distintos, e muitos estudantes têm exatamente o mesmo valor. **Regra adotada:** todos os estudantes com INSE **menor ou igual** ao percentil 25 são desfavorecidos, e todos com proficiência **maior ou igual** ao percentil 75 estão no quartil superior. Por causa dos empates, os grupos podem ter um pouco mais ou menos de 25% dos estudantes. A proporção exata é informada e usada no cálculo do valor esperado.

In [2]:
Q1_INSE = base['inse'].quantile(0.25)
Q3_PROF = base['proficiencia'].quantile(0.75)
base['desfavorecido'] = (base['inse'] <= Q1_INSE).astype(int)
base['quartil_sup_prof'] = (base['proficiencia'] >= Q3_PROF).astype(int)
base['resiliente'] = ((base['desfavorecido'] == 1) & (base['quartil_sup_prof'] == 1)).astype(int)

print(f'INSE: {base.inse.nunique()} valores distintos | percentil 25 = {Q1_INSE:.3f} | '
      f'estudantes exatamente nesse valor: {(base.inse == Q1_INSE).sum()}')
print(f'Proficiência: percentil 75 = {Q3_PROF:.4f} ({250 + FATOR_SAEB * Q3_PROF:.1f} pontos SAEB)')
print(f'Desfavorecidos: {base.desfavorecido.sum()} ({100 * base.desfavorecido.mean():.2f}% da base)')
print(f'Quartil superior de proficiência: {base.quartil_sup_prof.sum()} ({100 * base.quartil_sup_prof.mean():.2f}% da base)')

INSE: 475 valores distintos | percentil 25 = 4.540 | estudantes exatamente nesse valor: 90
Proficiência: percentil 75 = 1.0324 (306.9 pontos SAEB)
Desfavorecidos: 5804 (25.24% da base)
Quartil superior de proficiência: 5749 (25.00% da base)


## 2. Resiliência observada e esperada

Se o nível socioeconômico e a proficiência fossem independentes, a proporção de estudantes do quartil superior de proficiência entre os desfavorecidos seria igual à da base inteira.

In [3]:
d = base[base.desfavorecido == 1]
n_d, n_r = len(d), int(d.resiliente.sum())
taxa = n_r / n_d
p_marg = base.quartil_sup_prof.mean()
esperado = n_d * p_marg
lo, hi = ic_escolas(d.resiliente.values, d.id_escola.values)
taxa_pond = np.average(d.resiliente, weights=d.peso_aluno)
res = pd.DataFrame({'Indicador': ['Estudantes desfavorecidos', 'Estudantes resilientes', 'Taxa de resiliência (%)',
                                  'IC 95% da taxa (reamostragem de escolas)', 'Taxa ponderada pelo peso do aluno (%)',
                                  'Proporção no quartil superior na base inteira (%)', 'Resilientes esperados sob independência',
                                  'Razão observado / esperado', 'Resilientes na base inteira (%)'],
                    'Valor': [n_d, n_r, round(100 * taxa, 1), f'{100 * lo:.1f} a {100 * hi:.1f}'.replace('.', ','),
                              round(100 * taxa_pond, 1), round(100 * p_marg, 1), round(esperado, 0),
                              round(n_r / esperado, 3), round(100 * n_r / len(base), 1)]})
TABELAS['Resiliência'] = res
print(res.to_string(index=False))
# Proficiência média por faixa de INSE
Q3_INSE = base['inse'].quantile(0.75)
base['faixa_inse'] = np.select(
    [base['inse'] <= Q1_INSE, base['inse'] >= Q3_INSE],
    ['Desfavorecidos (25% menor INSE)', 'Favorecidos (25% maior INSE)'],
    default='Intermediários')

ordem = ['Desfavorecidos (25% menor INSE)', 'Intermediários', 'Favorecidos (25% maior INSE)']
faixas = (base.groupby('faixa_inse')['proficiencia_saeb']
              .agg(['size', 'mean', 'std']).reindex(ordem).round(1).reset_index())
faixas.columns = ['Grupo', 'Estudantes', 'Proficiência média (pontos SAEB)', 'Desvio-padrão']

# Diferença entre favorecidos e desfavorecidos, com erro-padrão agrupado por escola
dd = base[base['faixa_inse'] != 'Intermediários'].copy()
dd['favorecido'] = (dd['faixa_inse'] == ordem[2]).astype(int)
r = smf.ols('proficiencia ~ favorecido', dd).fit(
    cov_type='cluster', cov_kwds={'groups': dd['id_escola']}, use_t=True)
dif = r.params['favorecido'] * FATOR_SAEB
ic = r.conf_int().loc['favorecido'] * FATOR_SAEB
dp = dd.groupby('favorecido')['proficiencia_saeb'].std()
d_cohen = dif / np.sqrt((dp[0] ** 2 + dp[1] ** 2) / 2)

TABELAS['Proficiência por INSE'] = faixas
print(faixas.to_string(index=False))
print(f'\nDiferença favorecidos − desfavorecidos: {dif:.1f} pontos '
      f'(IC 95%: {ic[0]:.1f} a {ic[1]:.1f}; p {fmt_p(r.pvalues["favorecido"])}; d de Cohen = {d_cohen:.2f})')

                                        Indicador       Valor
                        Estudantes desfavorecidos        5804
                           Estudantes resilientes        1073
                          Taxa de resiliência (%)        18.5
         IC 95% da taxa (reamostragem de escolas) 17,1 a 19,7
            Taxa ponderada pelo peso do aluno (%)        18.6
Proporção no quartil superior na base inteira (%)        25.0
          Resilientes esperados sob independência      1451.0
                       Razão observado / esperado       0.739
                  Resilientes na base inteira (%)         4.7
                          Grupo  Estudantes  Proficiência média (pontos SAEB)  Desvio-padrão
Desfavorecidos (25% menor INSE)        5804                             260.4           48.6
                 Intermediários       11389                             270.4           49.9
   Favorecidos (25% maior INSE)        5803                             276.9           51.5

Diferen

## 3. Sensibilidade da definição

A taxa recalculada com outras definições.

- **Empates excluídos:** apenas INSE estritamente menor que o percentil 25.
- **Níveis nacionais do INSE:** desfavorecidos definidos pelos níveis I a III da escala nacional do INEP, e não pelo quartil da base. Esta versão aproxima o lado socioeconômico de uma referência nacional.
- **Casos completos:** apenas estudantes que responderam à escolaridade dos dois responsáveis.
- **Sem Ensino médio integrado:** apenas estudantes do Ensino Médio tradicional.

In [4]:
def calc(mask_desf, mask_base=None):
    b = base if mask_base is None else base[mask_base]
    md_ = mask_desf if mask_base is None else mask_desf[mask_base]
    q3 = b['proficiencia'].quantile(0.75)
    sup = b['proficiencia'] >= q3
    dd = sup[md_]
    return len(dd), int(dd.sum()), 100 * dd.mean(), 100 * sup.mean()

completos = ((base.nao_sabe_escolaridade_mae == 0) & (base.nao_sabe_escolaridade_pai == 0) &
             (base.ausente_escolaridade_mae == 0) & (base.ausente_escolaridade_pai == 0))
cenarios = {
    'Principal (INSE ≤ percentil 25)': (base.inse <= Q1_INSE, None),
    'Empates excluídos (INSE < percentil 25)': (base.inse < Q1_INSE, None),
    'Níveis nacionais I a III do INSE': (base.nivel_inse <= 3, None),
    'Casos completos na escolaridade': (base.inse <= base.loc[completos, 'inse'].quantile(0.25), completos),
    'Sem EM integrado': (base.inse <= base.loc[base.em_integrado == 0, 'inse'].quantile(0.25), base.em_integrado == 0),
}
linhas = []
for nome, (mdesf, mbase) in cenarios.items():
    n_d_, n_r_, tx, marg = calc(mdesf, mbase)
    linhas.append({'Definição': nome, 'Desfavorecidos': n_d_, 'Resilientes': n_r_, 'Taxa (%)': round(tx, 1),
                   'Esperado sob independência (%)': round(marg, 1), 'Razão obs./esp.': round(tx / marg, 3)})
sens = pd.DataFrame(linhas)
TABELAS['Sensibilidade'] = sens
print(sens.to_string(index=False))

                              Definição  Desfavorecidos  Resilientes  Taxa (%)  Esperado sob independência (%)  Razão obs./esp.
        Principal (INSE ≤ percentil 25)            5804         1073      18.5                            25.0            0.739
Empates excluídos (INSE < percentil 25)            5714         1052      18.4                            25.0            0.736
       Níveis nacionais I a III do INSE            5352          976      18.2                            25.0            0.729
        Casos completos na escolaridade            4176          784      18.8                            25.0            0.751
                       Sem EM integrado            5435         1021      18.8                            25.0            0.751


## 4. Taxas de resiliência por raça/cor e por sexo

O denominador é o número de estudantes de cada grupo dentro do quartil inferior.

In [13]:
linhas = []
for var, rot in [('raca_cor', 'Raça/cor'), ('sexo', 'Sexo')]:
    for g, dg in d.groupby(var):
        k, n = int(dg.resiliente.sum()), len(dg)
        wl, wh = proportion_confint(k, n, method='wilson')
        bl, bh = ic_escolas(dg.resiliente.values, dg.id_escola.values, B=500)
        linhas.append({'Variável': rot, 'Grupo': g, 'Desfavorecidos': n, 'Resilientes': k,
                       'Taxa (%)': round(100 * k / n, 1),
                       'IC 95% Wilson': f'{100 * wl:.1f} a {100 * wh:.1f}'.replace('.', ','),
                       'IC 95% por escolas': f'{100 * bl:.1f} a {100 * bh:.1f}'.replace('.', ','),})
taxas = pd.DataFrame(linhas)
TABELAS['Taxas por grupo'] = taxas
print(taxas.to_string(index=False))

Variável     Grupo  Desfavorecidos  Resilientes  Taxa (%) IC 95% Wilson IC 95% por escolas
Raça/cor   Amarela             150           15      10.0    6,2 a 15,8         5,6 a 15,3
Raça/cor    Branca            1175          284      24.2   21,8 a 26,7        22,0 a 26,7
Raça/cor  Indígena             108            5       4.6    2,0 a 10,4          0,9 a 9,4
Raça/cor     Parda            3376          591      17.5   16,3 a 18,8        16,0 a 18,9
Raça/cor     Preta             995          178      17.9   15,6 a 20,4        15,5 a 20,4
    Sexo  Feminino            3656          699      19.1   17,9 a 20,4        17,8 a 20,6
    Sexo Masculino            2148          374      17.4   15,9 a 19,1        15,5 a 19,5


## 5. Perfil dos estudantes resilientes

Comparação dentro do quartil inferior do INSE, entre resilientes e não resilientes. Médias e proporções usam apenas respostas válidas. Os valores-p vêm de regressões com erros-padrão agrupados por escola. por se tratar de uma análise descritiva com muitas comparações, informa-se também o valor-p ajustado por Benjamini-Hochberg. As variáveis de escolaridade e de posse de bens estão entre as que compõem o INSE e são marcadas na tabela.

In [6]:
VARS = {'aluno_motivacao_aplicar_aprendizado': 'Q23b (0 = concordo totalmente a 3 = discordo totalmente)',
        'tempo_estudos': 'Tempo de estudos (0 a 3)', 'tempo_cursos': 'Tempo em cursos (0 a 3)',
        'tempo_trabalho_domestico': 'Trabalho doméstico (0 a 3)', 'tempo_trabalho': 'Trabalho fora de casa (0 a 3)',
        'reprovacao': 'Reprovação (0 a 2)', 'idade': 'Idade (anos)',
        'responsaveis_incentivam_estudo': 'Responsáveis incentivam o estudo (0 a 2)',
        'responsaveis_conversam_escola': 'Responsáveis conversam sobre a escola (0 a 2)',
        'mora_mae': 'Mora com a mãe (%)', 'mora_pai': 'Mora com o pai (%)',
        'escolaridade_mae': 'Escolaridade da mãe (1 a 5)', 'escolaridade_pai': 'Escolaridade do pai (1 a 5)',
        'qtd_computadores': 'Computadores (0 a 3)', 'possui_wifi': 'Wi-Fi (%)',
        'possui_escrivaninha': 'Mesa para estudar (%)', 'possui_casa_quarto_individual': 'Quarto só seu (%)'}
INSE_ITENS = {'escolaridade_mae', 'escolaridade_pai', 'qtd_computadores', 'possui_wifi',
              'possui_escrivaninha', 'possui_casa_quarto_individual'}
d = d.assign(fem=(d.sexo == 'Feminino').astype(int), so_trab=(d.pretensao_futura == 'Somente trabalhar').astype(int),
             so_est=(d.pretensao_futura == 'Somente estudar').astype(int))
extras = {'fem': 'Sexo feminino (%)', 'so_est': 'Pretende somente estudar (%)',
          'so_trab': 'Pretende somente trabalhar (%)', 'em_integrado': 'EM integrado (%)'}

def validos(df, v):
    m = pd.Series(True, index=df.index)
    if 'ausente_' + v in df: m &= df['ausente_' + v] == 0
    if 'nao_sabe_' + v in df: m &= df['nao_sabe_' + v] == 0
    return df[m]

linhas = []
for v, rot in {**VARS, **extras}.items():
    dv = validos(d, v)
    r = smf.ols(f'{v} ~ resiliente', dv).fit(cov_type='cluster', cov_kwds={'groups': dv['id_escola']}, use_t=True)
    esc = 100 if '(%)' in rot else 1
    a, b = dv.loc[dv.resiliente == 1, v], dv.loc[dv.resiliente == 0, v]
    linhas.append({'Atributo': rot, 'Resilientes': a.mean() * esc, 'Não resilientes': b.mean() * esc,
                   'Diferença': (a.mean() - b.mean()) * esc, 'p-valor': r.pvalues['resiliente'],
                   'Componente do INSE': 'sim' if v in INSE_ITENS else ''})
perfil = pd.DataFrame(linhas)
perfil['p ajustado (BH)'] = multipletests(perfil['p-valor'], method='fdr_bh')[1]
perfil = perfil.round({'Resilientes': 3, 'Não resilientes': 3, 'Diferença': 3})
for c in ['p-valor', 'p ajustado (BH)']: perfil[c] = perfil[c].map(fmt_p)
TABELAS['Perfil'] = perfil
print(perfil.to_string(index=False))

                                                Atributo  Resilientes  Não resilientes  Diferença p-valor Componente do INSE p ajustado (BH)
Q23b (0 = concordo totalmente a 3 = discordo totalmente)        1.487            1.204      0.283 < 0,001                            < 0,001
                                Tempo de estudos (0 a 3)        1.513            1.254      0.258 < 0,001                            < 0,001
                                 Tempo em cursos (0 a 3)        1.092            0.844      0.249 < 0,001                            < 0,001
                              Trabalho doméstico (0 a 3)        2.207            2.098      0.109 < 0,001                            < 0,001
                           Trabalho fora de casa (0 a 3)        1.579            1.782     -0.203 < 0,001                            < 0,001
                                      Reprovação (0 a 2)        0.093            0.197     -0.104 < 0,001                            < 0,001
             

## 6. Modelo complementar

Regressão logística dentro do quartil inferior do INSE, da condição de resiliente sobre os mesmos atributos e controles do modelo principal, com erros-padrão agrupados por escola. Mostra quais características se associam à resiliência quando consideradas em conjunto. Razões de chances acima de 1 indicam maior chance de o estudante desfavorecido estar no quartil superior de proficiência.

In [16]:
ATRIB = ("C(sexo, Treatment('Masculino')) + C(raca_cor, Treatment('Branca')) + "
         "C(pretensao_futura, Treatment('Somente estudar')) + " + ' + '.join(VARS))
CONTROLES = ('em_integrado + nao_sabe_escolaridade_mae + nao_sabe_escolaridade_pai + ausente_escolaridade_mae + '
             'ausente_escolaridade_pai + ausente_aluno_motivacao_aplicar_aprendizado + ausente_mora_mae + '
             'ausente_mora_pai + n_itens_ausentes')
logit = smf.logit('resiliente ~ ' + ATRIB + ' + ' + CONTROLES, d).fit(
    disp=False, cov_type='cluster', cov_kwds={'groups': d['id_escola']})
ic = logit.conf_int()
termos = [t for t in logit.params.index if t != 'Intercept' and not any(c in t for c in CONTROLES.split(' + '))]
lg = pd.DataFrame({'Termo': termos,
                   'Razão de chances': np.exp(logit.params[termos]).round(3),
                   'IC 95%': [f'{np.exp(ic.loc[t, 0]):.2f} a {np.exp(ic.loc[t, 1]):.2f}'.replace('.', ',') for t in termos],
                   'p-valor': [fmt_p(logit.pvalues[t]) for t in termos]})
lg['Termo'] = lg['Termo'].str.replace(r"C\((\w+), Treatment\('[^']+'\)\)\[T\.(.+)\]", r'\1: \2', regex=True)
TABELAS['Modelo logístico'] = lg
print(f'N = {int(logit.nobs)} | pseudo-R² de McFadden = {logit.prsquared:.3f}')
print(lg.to_string(index=False))

N = 5804 | pseudo-R² de McFadden = 0.107
                                Termo  Razão de chances      IC 95% p-valor
                       sexo: Feminino             0.851 0,73 a 1,00   0,047
                    raca_cor: Amarela             0.441 0,26 a 0,76   0,003
                   raca_cor: Indígena             0.205 0,08 a 0,55   0,002
                      raca_cor: Parda             0.761 0,64 a 0,90   0,002
                      raca_cor: Preta             0.780 0,63 a 0,97   0,027
     pretensao_futura: Ainda não sabe             0.861 0,57 a 1,30   0,481
pretensao_futura: Estudar e trabalhar             1.422 1,00 a 2,02   0,049
  pretensao_futura: Somente trabalhar             0.533 0,34 a 0,83   0,006
  aluno_motivacao_aplicar_aprendizado             1.599 1,45 a 1,77 < 0,001
                        tempo_estudos             1.345 1,22 a 1,48 < 0,001
                         tempo_cursos             1.133 1,05 a 1,22 < 0,001
             tempo_trabalho_domestico          

## Exportação

In [8]:
with pd.ExcelWriter(PASTA + 'tabelas_resiliencia.xlsx') as xw:
    for nome, t in TABELAS.items():
        t.to_excel(xw, sheet_name=nome[:31], index=False)
print('Tabelas salvas em', PASTA + 'tabelas_resiliencia.xlsx')

Tabelas salvas em /content/drive/MyDrive/dissertacao/tabelas_resiliencia.xlsx
